# 05 · Live training curves from the log file

Training (notebook 04 / `scripts/train.py`) appends every metric to
`logs/metrics.jsonl`. This notebook **tails that file**, so you can run it in
parallel *while training is still happening* and re-execute the plot cell (or
use the auto-refresh cell) for an updated picture. Plots are also saved as
PNGs under `logs/` — making the plots themselves part of the run's artifacts.

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

def load_metrics(path=None):
    path = Path(path or cfg.paths.metrics_file)
    if not path.exists():
        raise FileNotFoundError(f"{path} — start training first (notebook 04)")
    evs = [json.loads(l) for l in open(path) if l.strip()]
    by = lambda e: [x for x in evs if x["event"] == e]
    return {"train": by("train_step"), "val": by("val"),
            "epoch": by("epoch"), "skip": by("skip")}

m = load_metrics()
{k: len(v) for k, v in m.items()}

## 1. Train & validation loss

In [ ]:
def plot_losses(m, save=True):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if m["train"]:
        axes[0].plot([e["step"] for e in m["train"]],
                     [e["loss"] for e in m["train"]], lw=0.8, label="train (step)")
        axes[0].set_xlabel("optimizer step"); axes[0].set_yscale("log")
        axes[0].set_title("train loss"); axes[0].legend()
    if m["epoch"]:
        ep = [e["epoch"] for e in m["epoch"]]
        axes[1].plot(ep, [e["train_loss"] for e in m["epoch"]], "o-", label="train (epoch)")
        vl = [(e["epoch"], e["val_loss"]) for e in m["epoch"] if e["val_loss"] is not None]
        if vl:
            axes[1].plot(*zip(*vl), "s-", label="val")
            best = min(v for _, v in vl)
            axes[1].axhline(best, color="green", ls=":", label=f"best val {best:.3f}")
        axes[1].set_xlabel("epoch"); axes[1].set_title("epoch losses"); axes[1].legend()
    plt.tight_layout()
    if save:
        out = Path(cfg.paths.logs_dir) / "loss_curves.png"
        fig.savefig(out, dpi=140); print("saved →", out)
    plt.show()

plot_losses(m)

## 2. Learning rate, gradient norm, skipped steps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
steps = [e["step"] for e in m["train"]]
axes[0].plot(steps, [e["lr"] for e in m["train"]]); axes[0].set_title("LR schedule")
axes[1].plot(steps, [e["grad_norm"] for e in m["train"]], lw=0.7)
axes[1].axhline(cfg.training.grad_clip_norm, color="red", ls="--", label="clip")
axes[1].set_title("grad norm (pre-clip)"); axes[1].legend()
sk = m["skip"]
axes[2].step([s["ts"] for s in sk], range(1, len(sk)+1)) if sk else axes[2].text(.5,.5,"no skipped steps ✔", ha="center")
axes[2].set_title(f"skipped steps (total {len(sk)})")
plt.tight_layout()
fig.savefig(Path(cfg.paths.logs_dir) / "diagnostics.png", dpi=140)
plt.show()

## 3. Auto-refresh while training runs (stop with ⏹)

In [ ]:
import time
from IPython.display import clear_output
REFRESH_SEC, ROUNDS = 30, 20          # ~10 minutes of watching
for _ in range(ROUNDS):
    clear_output(wait=True)
    try:
        plot_losses(load_metrics(), save=True)
    except FileNotFoundError as e:
        print(e)
    time.sleep(REFRESH_SEC)